# P1 — RSNA Knee: reproducir todo el pipeline en Kaggle

**Configuración del notebook** (panel derecho → *Settings* / *Input*), una sola vez:

1. **Accelerator: GPU T4 x1.** No usar P100: las versiones recientes de PyTorch ya no la soportan.
2. **Internet: ON.** Hace falta para clonar el repositorio y descargar los pesos ImageNet (ResNet-50) y el modelo NLI. El texto de los reportes no sale del notebook.
3. **Add Input → Competitions → `rsna-knee-abnormality-detection`.** Tu cuenta debe haber aceptado las *Rules* de la competencia.

Luego **Run All**. Duración de la ejecución oficial (30-oct-2026, T4): **≈ 36 min**.
Las salidas quedan en `/kaggle/working/p1/` y al final se comparan contra los resultados versionados en el repositorio.

In [ ]:
# 0. Entorno
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
import glob
print(glob.glob('/kaggle/input/*/*'))

## 1. Obtener el código

In [ ]:
REPO_URL = "https://github.com/<USUARIO>/<REPO>.git"   # <-- URL del repositorio del grupo
!rm -rf /kaggle/working/repo
!git clone -q --depth 1 $REPO_URL /kaggle/working/repo
!ls /kaggle/working/repo/03_scripts

## 2. Ejecutar el pipeline completo (comando único)

Pasos 01 → 08: inventario, etiquetas débiles (reglas + NLI opcional), validación contra dorados, subconjunto,
caché DICOM 2.5D, 7 entrenamientos (A1, A2, B1–B3, C1, C2), Grad-CAM y tablas.

In [ ]:
!bash /kaggle/working/repo/03_scripts/run_all.sh

## 3. Comparar con la ejecución oficial versionada en el repositorio

In [ ]:
!python /kaggle/working/repo/tools/verificar_resultados.py --oficial /kaggle/working/repo --nuevo /kaggle/working/p1

## 4. Ver tablas y figuras

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('/kaggle/working/p1/06_resultados/08_comparacion_runs.md').read()))
for f in sorted(glob.glob('/kaggle/working/p1/06_resultados/figuras/*.png')):
    print(f); display(Image(f, width=700))

## 5. Descargar resultados (sin caché ni pesos)

In [ ]:
%cd /kaggle/working/p1
!zip -qr /kaggle/working/p1_resultados.zip 02_logs 06_resultados -x "*.pt"
!ls -lh /kaggle/working/p1_resultados.zip